# Initialization

## Path configuration

In [ ]:
# Import required libraries
import sys
from pathlib import Path

# Navigate up from 'src/notebooks/' to the project root directory
PROJECT_ROOT = Path('..').resolve().parent
DATA_DIR = PROJECT_ROOT / 'data'
SRC_DIR = PROJECT_ROOT / 'src'

# Add src to python path for imports
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# Enable automatic reloading (avoid restarting kernel when editing internal modules)
%reload_ext autoreload
%autoreload 2

## Library imports

In [ ]:
# General imports
import os
from random import random
import numpy as np
import pandas as pd

# Neural networks
import tensorflow as tf
from tensorflow import keras as ks

# Binary classification
from sklearn.metrics import confusion_matrix

# Principal component analysis
from sklearn.decomposition import PCA

# Persistent homology
from ripser import ripser

# Plotting
import plotly.io as pio

# Local imports
from utils.plots import *

In [ ]:
# Force Keras to use CPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Set renderer for plots
pio.renderers.default = "plotly_mimetype+notebook_connected"

## Auxiliary functions

In [ ]:
def apply_weights(model, train_data):
    num_layers = len(model.layers) - 1  # We consider only the hidden layers

    # Initialize with the input data
    trans_train_data = [train_data]

    # Apply successive transformations through layers
    for i in range(num_layers):
        # Get components to apply the affine transformation
        linear_part = model.layers[i].get_weights()[0]
        affine_part = model.layers[i].get_weights()[1]
        last = trans_train_data[-1]

        # Apply the affine transformation
        transformed = np.dot(last, linear_part) + affine_part

        # Apply ReLU activation
        activated = ks.layers.ReLU()(transformed)

        # Store the result
        trans_train_data.append(activated)

    return np.array(trans_train_data[1:])  # Drop the training data

In [ ]:
def compute_sigmoid_output(model, trans_train_data):
    linear_part = model.layers[-1].get_weights()[0]
    affine_part = model.layers[-1].get_weights()[1]
    last = trans_train_data[-1]

    transformed = np.dot(last, linear_part) + affine_part

    # We directly transform the output probabilities into a binary result
    activated = tf.cast(tf.sigmoid(transformed) > 0.5, tf.float32)

    return np.array(activated[:,0])

In [ ]:
def apply_pca(trans_train_data, dim=2):
    pca = PCA(n_components=dim)
    pca_trans_train_data = [pca.fit_transform(h) for h in trans_train_data]

    return np.array(pca_trans_train_data)

# Training data

In [ ]:
# Generate training data
def add_traning_data(dcls):
    # Initialize lists to store the data points and their class
    train_data = []
    train_labels = []

    for _ in range(10000):
        # Generate a random point in a circle
        r = 4 * random()
        a = 2 * np.pi * random()
        x = r * np.cos(a)
        y = r * np.sin(a)
        train_data.append(np.array([x, y]))

        # Assign to class A or B according to some conditions
        cond_left_eye = (x-np.sqrt(2))**2 + (y-np.sqrt(2))**2 < 1
        cond_right_eye = (x+np.sqrt(2))**2 + (y-np.sqrt(2))**2 < 1
        cond_mouth = (x/2)**2 + (y+2)**2 < 1
        cond_nose = x**2 + y**2 < 0.1
        train_labels.append(int(cond_left_eye or cond_right_eye or cond_mouth or cond_nose))

    # Transform to numpy arrays
    train_data = np.array(train_data)
    train_labels = np.array(train_labels)

    # Store the training data
    dcls["A"]["td"] = train_data[np.where(train_labels==0)]
    dcls["B"]["td"] = train_data[np.where(train_labels==1)]

    # Define the color of each class
    dcls["A"]["col"] = "red"
    dcls["B"]["col"] = "green"

    return dcls, train_data, train_labels


DCLS = {"A":{}, "B":{}}
DCLS, train_data, train_labels = add_traning_data(DCLS)

We created a dataset with two classes A (in <span style="color:red">**red**</span>) and B (in <span style="color:green">**green**</span>). We choose a very recognizable shape in order to facilitate visualizing the results:

- Class A is contained within a circle of radius 2.

- Class B determines four isolated "islands" within the circle.

In [ ]:
plot_training_data(DCLS)

# Neural network

## Create and train

In [ ]:
MODEL = ks.Sequential([
    ks.layers.Dense(16, activation="relu"),  # Layer 1
    ks.layers.Dense(16, activation="relu"),  # Layer 2
    ks.layers.Dense(16, activation="relu"),  # Layer 3
    ks.layers.Dense(16, activation="relu"),  # Layer 4
    ks.layers.Dense(16, activation="relu"),  # Layer 5
    ks.layers.Dense(16, activation="relu"),  # Layer 6
    ks.layers.Dense(16, activation="relu"),  # Layer 7
    ks.layers.Dense(16, activation="relu"),  # Layer 8
    ks.layers.Dense(16, activation="relu"),  # Layer 9
    ks.layers.Dense(16, activation="relu"),  # Layer 10
    ks.layers.Dense(16, activation="relu"),  # Layer 11
    ks.layers.Dense(16, activation="relu"),  # Layer 12
    ks.layers.Dense(16, activation="relu"),  # Layer 13
    ks.layers.Dense(16, activation="relu"),  # Layer 14
    ks.layers.Dense(16, activation="relu"),  # Layer 15
    ks.layers.Dense(1, activation="sigmoid")
])

In [ ]:
MODEL.compile(
    optimizer=ks.optimizers.Adam(),
    loss=ks.losses.BinaryCrossentropy(),
    metrics=[
        ks.metrics.BinaryAccuracy(),
        ks.metrics.FalseNegatives(),
        ks.metrics.FalsePositives()
    ]
)

We created a neural network with:

- 15 dense layers (16 neurons each, ReLU activation)

- A final layer with a single neuron and sigmoid activation to produce a binary classifaction

It is important to have a large number of intermediate layers to observe how the data shape changes as it passes through the layers.

We fit the neural network to the training data:

In [ ]:
MODEL.fit(train_data, train_labels, epochs=5, batch_size=1)

## Transformed data points

We now store the training set data points as they are transformed by each of the layers of the neural network. Since the intermediate layers have 16 neurons each, data points live in a 16-dimensional space.

In [ ]:
def transform_data_points(model, dcls):
    for cs in dcls:
        dcls[cs]["ttd"] = apply_weights(model, dcls[cs]["td"])
        dcls[cs]["out"] = compute_sigmoid_output(model, dcls[cs]["ttd"])
        dcls[cs]["pttd"] = apply_pca(dcls[cs]["ttd"])

    return dcls

DCLS = transform_data_points(MODEL, DCLS)

In order to visualize it we use PCA to project the result into 2-dimensional spaces that capture the separations as well as possible.

In [ ]:
plot_side_by_side(
    plot_data_shape_evolution(DCLS, "A", MODEL),
    plot_data_shape_evolution(DCLS, "B", MODEL)
)

As we move the sliders to the right we see how the data progressively loses its original shape and bends over itself up to the point where, at the end it defines a very clear single 1-dimensional shape. Note also how the x-axis stretches as we take new steps.

## Classification of transformed points

We now show visually how, after the data is transformed under the neural network, it classifies the results. To do that we take the final transformed data and for each true class we paint the transformed points according to what the model predicts.

In [ ]:
plot_side_by_side(
    plot_classification(DCLS, "A"),
    plot_classification(DCLS, "B")
)

We can observe that for both A and B, once the shape of the data is transformed, the sigmoid classifier is perfectly able to separate the two classes and that, as expected, the correct class is much more dense.

Below we can confirm with the confusion matrix that the model did a great job at separating the two classes.

In [ ]:
def compute_confusion_matrix(dcls):
    array_0 = dcls["A"]["out"]
    array_1 = dcls["B"]["out"]

    true = np.concatenate([np.zeros(len(array_0)), np.ones(len(array_1))])
    pred = np.concatenate([array_0, array_1])

    cm = confusion_matrix(true, pred)

    df = pd.DataFrame(
        columns=["Predicted A", "Predicted B"],
        index=["True class A", "True class B"],
        data=cm
    )

    return df

compute_confusion_matrix(DCLS)

# Intro to Betti numbers and persistent homology

## Betti numbers

Betti numbers are a mathematical concept from the area of Topology. They encode some characteristics of shape:

- The 0th Betti number is the number of connected components of the shape

- The 1st Betti number counts the number of holes in the shape

- The 2nd Betti number counts the number of bubbles (a "hole" enclosed by a sphere)

- In general, the n-th Betti number counts the number of "(n+1)-dimensional holes" in the shape

In our example, our data is contained in a plane, so the only Betti numbers that are relevant are the 0th and the 1st. Let us recall the shape of our training data and describe the Betti numbers we should expect.

In [ ]:
plot_side_by_side(
    plot_class(DCLS, "A"),
    plot_class(DCLS, "B")
)

- Class A has a single connected component and four holes, so we expect its Betti numbers to be (1, 4)
- Class B has four connected components and no holes, so we expect their Betti numbers to be (4, 0)

## Persistent homology

Persistent homology is a technique that given a cloud of points it computes the most likely Betti numbers of the shape they conform.

In loose terms, the main idea consists in drawing a small disk (or sphere) around each circle and count the holes that persist as we keep making those disks larger and they start covering more and more space:

- A hole that is covered quickly (i.e. is not persistent) can be attributed to noise in the data

- A hole that persists a long time can be regarded as a true area where there are no data points

It is not our goal to give a full account of persistent homology. Those interested can check this interesting video on the topic: [https://www.youtube.com/watch?v=SbsvM4Gcbl0](https://www.youtube.com/watch?v=SbsvM4Gcbl0)

We shall now see visually the main technique of persistent homology and the expected Betti numbers that we mentioned should become apparent.

In [ ]:
plot_side_by_side(
    plot_increasing_disks(DCLS, "A"),
    plot_increasing_disks(DCLS, "B")
)

### Evaluating persistent homology

In [ ]:
def prepare_persistent_homology_diagrams(dcls):
    for cs in ["A", "B"]:
        dcls[cs]["td_diagrams"] = ripser(dcls[cs]["td"])["dgms"]
        dcls[cs]["last_pttd_diagrams"] = ripser(dcls[cs]["pttd"][-1])["dgms"]

    return dcls

DCLS = prepare_persistent_homology_diagrams(DCLS)

As a final addition we show the birth-death scatters that are commonly used to evaluate persistent homology. It is not our goal to provide a very detailed account of it and we refer to the video above for more details in this topic.

How to read the data?

- Each diagram contains two rows: one for the training data and one for the transformed data

- For each row we show the data and the birth-death scatter: the 0-dim scatter (0th Betti number) in blue color and the 1-dim scatter (1st Betti number) in orange color.

- In birth-death scatter any point near the diagonal is a feature that doesn't persist and hence it is not relevant

- Relevant features are the ones that are far from the diagonal

- Note that 0-dim diagrams always show a dot at infinity. This accounts for the fact that when the disks keep growing eventually all of them superpose and form a single connected component (remember this is what 0th Betti number counts) that never vanishes.

#### Class A

In [ ]:
plot_persistent_homology(DCLS, "A")

Training data:

  - In the persistent homology diagram we can see how most of the 0-dim points die soon except for the expected dot at infinity, indicating a single connected component: this confirms the 0th Betti number is 1.

  - For the 1-dim points (in orange) we can see how four of them are far from the diagonal. These account for the four holes in the data cloud. Among these four there is one that dies first (the "nose"), another two that die almost at the same time (the "eyes") and one that dies latest (the "mouth") according each hole size.

Transformed data:

  - There are some 0-dim elements that take some time to die. In effect, if we look at the data we see some quite isolated points which naturally take time until their sorrounding disk is large enough to connect to another disk (moment in which it dies as a distinguished connected component).

  - Moreover, remember that the transformed data is just a 2-dimensional representation of a 16-dimensional space so, in reality, the points are much more isolated and the curse of dimensionality applies.

  - The fact that very few (if any) point appears for the 1-dim diagram and that, if they do, they day immediately indicates the fact that, once transformed, the data doesn't contain any shape that can minimally resemble a hole.

#### Class B

In [ ]:
plot_persistent_homology(DCLS, "B")

Training data:

  - For 0-dim there are a few points -it is actually three- that die late. These together with the point at infinity account for the four connected components (0th Betti number is 4).

  - The fact that all three features die almost at the same time has a meaning: looking at the shape we see that the "eyes" and the "mouth" are at a similar distance from the "nose", so the disk size required for each of these components to merge with each one another is similar.

  - All the 1-dim points are close to the diagonal meaning there are no relevant holes: 1st Betti number is 0.

Transformed data:

  - Same comments we made for class A apply here.

# Final Takeaways

- The complexity of point cloud shapes can be encoded through Betti numbers

- Betti numbers can be computed with persistent homology

- Each new layer on a neural network simplifies Betti numbers (i.e. the data shape)

- Once shape is simplified, it is easier to perform some tasks, such as biary classification